# Plan-Then-Act Demo: Web Search + Calculator

This notebook drives `adk.demos.plan_then_act_demo.DemoPlanThenActAgent`, a concrete
`PlannerExecutorBase` subclass built on the **plan-then-act** pattern from `adk.planner_executor`.
It wires two real, heterogeneous tools behind fault-tolerant executor boundaries:

- **`web_search`** — a live Tavily web search
- **`calculate`** — a sandboxed arithmetic evaluator (no `eval()`; an AST allowlist, since the
  expression can be LLM-influenced input)

The agent's own code lives in `src/adk/demos/plan_then_act_demo.py` — this notebook imports it
and walks through what it does, rather than redefining any of the logic here.

**This demo is intentionally contrived.** The task below is small and predictable on purpose,
so the thing on display is the *plumbing* — real tool calls crossing two heterogeneous
executors, dependency-wave scheduling, degraded-mode handling, the plan-then-act topology
itself, all running against live APIs instead of fakes. It is not meant to show off planning
sophistication or anything worth calling reasoning ability; later examples, built on the same
base classes, will lean into tasks that actually exercise that.

## Setup

Requires two API keys:

- `ANTHROPIC_API_KEY` — https://console.anthropic.com/
- `TAVILY_API_KEY` — https://tavily.com/

Copy `.env.example` (repo root) to `.env` and paste your keys in there — `load_dotenv()`
below loads it into this process. `.env` is gitignored, so real keys never get committed.
`adk.anthropic_client.get_anthropic_client()` and `adk.search_client.get_search_client()`
raise a clear `RuntimeError` if either key is still missing.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

for key in ("ANTHROPIC_API_KEY", "TAVILY_API_KEY"):
    print(f"{key}: {'set' if os.environ.get(key) else 'MISSING'}")

## The tools

Both tools share one signature — `(args: dict, ctx: dict) -> Any` — the contract
`DegradedModeExecutor` expects: plain callables, agnostic to how they're implemented.

In [ ]:
from adk.demos.plan_then_act_demo import (
    DEFAULT_TASK,
    DemoPlanThenActAgent,
    calculate,
    web_search,
)

### `calculate` — safe by construction

`calculate` parses the expression with `ast.parse(..., mode="eval")` and walks the tree,
allowing only numeric constants, `+ - * / // % **`, and unary `+ -`. Anything else — a `Name`,
a `Call`, an `Attribute` — is rejected. Try a normal expression, then an injection attempt:

In [ ]:
print(calculate({"expression": "68000000 + 84000000"}, {}))

try:
    calculate({"expression": "__import__('os').system('echo pwned')"}, {})
except ValueError as exc:
    print(f"blocked: {exc}")

### `web_search` — live Tavily search

Calls `get_search_client().search(...)` and maps the top results down to
`{"title", "url", "snippet"}`. This cell makes a real API call.

In [ ]:
result = web_search({"query": "current population of France"}, {})
result

## Why plan-then-act (and its tradeoff)

`build_plan_then_act_graph` compiles: `produce_plan -> execute_plan -> draft_response -> END`.
The planner produces the **entire** plan up front, before any tool has run — there's no
mechanism to feed one step's tool output into a later step's arguments. Compare that to the
*interleaved* pattern (`build_planner_executor_graph`), which re-plans after every tool call and
so *can* thread live results forward.

The demo task below — "look up the current population of France and Germany, then calculate
their combined population" — is chosen for how legible it makes the *orchestration*, not for
planning difficulty:

- Two independent `search` steps run concurrently (no `depends_on` between them).
- The `calc` step's `depends_on` orders it after both searches, exercising the execution
  coordinator's dependency-wave scheduling.
- The planner has to correctly route each step to the executor that owns its tool.

That's routing and ordering, not reasoning about the task itself. What the demo does *not* do
is compute the sum from the live search numbers — the `calculate` step's expression is filled
in from the planner LLM's own knowledge at plan time. That's an intentional pattern tradeoff,
not a bug: true tool-output-to-tool-input chaining is the interleaved pattern's job, not
plan-then-act's. A future example can pick a task where that tradeoff actually bites.

## Building the agent

`DemoPlanThenActAgent._build_graph()` assembles:

- **planner** — `build_plan_then_act_planner(client, model=, system_prompt=PLANNER_SYSTEM_PROMPT)`,
  an `AnthropicRunnable` forced to call the `output_plan` tool so its response always parses into
  a `PlanThenActArtifact`.
- **executors** — `{"search": DegradedModeExecutor(...), "calc": DegradedModeExecutor(...)}`,
  each wrapping one tool and degrading gracefully (instead of failing the whole run) if its tool
  raises.
- **drafter** — a plain `AnthropicRunnable` that synthesizes the accumulated observations into a
  final answer.
- **config** — `PlannerExecutorConfig(tools_by_executor={"search": ("web_search",), "calc":
  ("calculate",)})`, the routing fallback the execution coordinator uses if a step's
  `executor_id` doesn't directly match a key in `executors`.

Constructing the agent builds and compiles the graph immediately (`PlannerExecutorBase.__init__`
calls `_build_graph()`), so this cell needs a valid `ANTHROPIC_API_KEY` to create the underlying
client.

In [ ]:
agent = DemoPlanThenActAgent()
agent.get_compiled_graph().get_graph().print_ascii()

## Running the demo task

`.invoke()` maps `{"task", "context"}` to initial graph state, runs the compiled graph, and
attaches an `invoke_trace` (run id, variant, prompt manifest) to the output.

In [ ]:
out = agent.invoke({"task": DEFAULT_TASK, "context": {}})
print(f"Task: {DEFAULT_TASK}")

### Authorized plan

The steps the planner produced, in order, with their dependencies:

In [ ]:
def print_plan(steps):
    for i, step in enumerate(steps):
        deps = step.depends_on
        print(f"{i}. [{step.executor_id}] {step.tool_name}({step.tool_args}) depends_on={deps}")


print_plan(out["authorized_steps"])

### Executed steps

What each executor actually did (`status` is `"ok"` or `"degraded"`):

In [ ]:
for step in out["executed_steps"]:
    print(step)

### Final draft answer

The drafter's synthesis of all observations into a direct answer:

In [ ]:
print(out["draft_output"]["text"])

## Try your own task

Swap in any task that plausibly needs one or both tools — the planner decides which
executor(s) to use.

In [ ]:
custom_task = "What is 17 times the number of member states in the EU?"

custom_out = agent.invoke({"task": custom_task, "context": {}})
print_plan(custom_out["authorized_steps"])
print()
print(custom_out["draft_output"]["text"])

## Trace metadata

`invoke_trace` carries a fresh `run_id`, the agent's `variant`, and its `prompt_manifest` — the
shape `PlannerExecutorBase.to_eval_run_result()` consumes to build an eval-harness `RunResult`
row from this same output.

In [ ]:
out["invoke_trace"]